# Interactive Agent Session: Chapter 2 — LangGraph Customer Support Router

This Jupyter notebook provides a gorgeous sandbox to watch **LangGraph** deterministically route customer support tickets through a strict state-machine graph. An `Intent Classifier` node reads the ticket, a **Routing Edge** enforces the correct path, and either a `Refund Agent` or a `Technical Agent` handles the resolution.

**Prerequisites:** Set your `OPENAI_API_KEY` below before running. The notebook gracefully falls back to simulated responses if no key is provided.

In [ ]:
import sys
!{sys.executable} -m pip install --quiet langchain langchain_openai langgraph


In [ ]:
import os
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from IPython.display import display, HTML

# ── 1. API Key ────────────────────────────────────────────────────────────────
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# ── 2. Define the shared state passed between every graph node ────────────────
class SupportState(TypedDict):
    ticket_text: str
    intent: str
    response: str
    actions: list
    log: list          # rich chat history for the UI renderer

llm = ChatOpenAI(model="gpt-4-turbo")

# ── 3. Node: Intent Classifier ───────────────────────────────────────────────
def classify_intent_node(state: SupportState):
    prompt = (
        f"You are a customer support triage system. "
        f"Determine the intent of this ticket: '{state['ticket_text']}'. "
        f"Output EXACTLY one word — either 'refund' or 'technical'."
    )
    try:
        result = llm.invoke(prompt).content.strip().lower()
        intent = "refund" if "refund" in result else "technical"
    except Exception:
        intent = "refund"   # demo fallback
    msg = f"🔍 Ticket received. Intent classified as **{intent.upper()}**."
    return {
        "intent": intent,
        "actions": ["classified"],
        "log": state["log"] + [{"sender": "Intent_Classifier", "msg": msg}]
    }

# ── 4. Node: Refund Agent ────────────────────────────────────────────────────
def refund_agent_node(state: SupportState):
    # In production this would call: requests.post('api.stripe.com/v1/refunds', ...)
    response = (
        "✅ Refund approved! I have initiated a full refund to your original payment method. "
        "You should see the credit within 3–5 business days. "
        "A confirmation email has been dispatched to the address on file."
    )
    return {
        "response": response,
        "actions": state["actions"] + ["refunded"],
        "log": state["log"] + [{"sender": "Refund_Agent", "msg": response}]
    }

# ── 5. Node: Technical Support Agent ────────────────────────────────────────
def technical_agent_node(state: SupportState):
    prompt = (
        f"You are a senior technical support engineer. "
        f"Draft a polite, professional troubleshooting email for this issue: '{state['ticket_text']}'. "
        f"Keep it under 80 words and include a concrete first step for the customer to try."
    )
    try:
        response = llm.invoke(prompt).content
    except Exception:
        response = (
            "📧 Hi there! Thank you for reaching out. "
            "Please try clearing your browser cache (Ctrl+Shift+Delete) and restarting the app. "
            "If the issue persists, reply with your OS version and we will escalate immediately. — Support Team"
        )
    return {
        "response": response,
        "actions": state["actions"] + ["drafted_email"],
        "log": state["log"] + [{"sender": "Technical_Agent", "msg": response}]
    }

# ── 6. Routing Edge ──────────────────────────────────────────────────────────
def route_ticket(state: SupportState) -> Literal["refund_agent", "technical_agent"]:
    return "refund_agent" if state["intent"] == "refund" else "technical_agent"

# ── 7. Compile the Graph Architecture ────────────────────────────────────────
workflow = StateGraph(SupportState)
workflow.add_node("classifier", classify_intent_node)
workflow.add_node("refund_agent", refund_agent_node)
workflow.add_node("technical_agent", technical_agent_node)

workflow.set_entry_point("classifier")
workflow.add_conditional_edges("classifier", route_ticket)
workflow.add_edge("refund_agent", END)
workflow.add_edge("technical_agent", END)

app = workflow.compile()

# ── 8. Beautiful Glassmorphism UI ─────────────────────────────────────────────
def render_support_chat(log: list, ticket: str):
    html = '''
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;800&display=swap');
        .sc-wrap { max-width: 820px; margin: 24px auto; font-family: 'Inter', sans-serif; background: #0f0f1a; padding: 32px; border-radius: 24px; box-shadow: 0 24px 60px rgba(0,0,0,0.6); border: 1px solid #1e1e3a; }
        .sc-title { text-align:center; font-size: 22px; font-weight: 800; letter-spacing: 2px; text-transform: uppercase; margin-bottom: 8px; background: linear-gradient(90deg, #a78bfa, #60a5fa); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
        .sc-subtitle { text-align:center; font-size: 13px; color: #555; margin-bottom: 28px; letter-spacing: 0.5px; }
        .sc-ticket { background: #1a1a2e; border-left: 4px solid #6366f1; padding: 16px 20px; border-radius: 12px; color: #d1d5db; font-size: 14px; margin-bottom: 24px; line-height: 1.6; }
        .sc-ticket strong { color: #a78bfa; font-size: 11px; text-transform: uppercase; letter-spacing: 1px; display: block; margin-bottom: 6px; }
        .sc-bubble { padding: 16px 22px; margin: 16px 0; border-radius: 18px; max-width: 85%; font-size: 14px; line-height: 1.7; color: #fff; box-shadow: 0 6px 20px rgba(0,0,0,0.2); transition: transform 0.15s; }
        .sc-bubble:hover { transform: translateY(-2px); }
        .sc-name { font-size: 11px; font-weight: 800; text-transform: uppercase; letter-spacing: 1.5px; margin-bottom: 8px; opacity: 0.85; }
        .Intent_Classifier { background: linear-gradient(135deg, #7c3aed, #4f46e5); margin-right: auto; border-bottom-left-radius: 4px; border: 1px solid #4f46e5; }
        .Refund_Agent      { background: linear-gradient(135deg, #059669, #10b981); margin-right: auto; border-bottom-left-radius: 4px; border: 1px solid #059669; }
        .Technical_Agent   { background: linear-gradient(135deg, #d97706, #f59e0b); margin-right: auto; border-bottom-left-radius: 4px; border: 1px solid #d97706; color: #0a0a0a !important; }
        .sc-actions { margin-top: 24px; padding: 14px 20px; background: #111827; border-radius: 12px; border: 1px solid #1f2937; }
        .sc-actions p { margin: 0; color: #6b7280; font-size: 12px; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; }
        .sc-tag { display: inline-block; background: #1e293b; color: #60a5fa; padding: 4px 12px; border-radius: 20px; font-size: 12px; font-weight: 600; margin: 6px 4px 0 0; border: 1px solid #334155; }
    </style>
    <div class="sc-wrap">
        <div class="sc-title">⚡ LangGraph Support Router</div>
        <div class="sc-subtitle">Deterministic State-Machine Routing — Chapter 2 Demo</div>
    '''
    html += f'<div class="sc-ticket"><strong>📨 Incoming Ticket</strong>{ticket}</div>'

    actions_collected = []
    for entry in log:
        sender_raw = entry.get("sender", "System")
        sender_cls = sender_raw.replace(" ", "_")
        icon = {"Intent_Classifier": "🤖", "Refund_Agent": "💳", "Technical_Agent": "🛠"}.get(sender_cls, "💬")
        html += f'''
        <div class="sc-bubble {sender_cls}">
            <div class="sc-name">{icon} {sender_raw.replace("_", " ")}</div>
            <div>{entry.get("msg", "")}</div>
        </div>'''

    html += '</div>'
    display(HTML(html))

# ── 9. Run Two Example Tickets ───────────────────────────────────────────────
tickets = [
    "I was charged twice for my subscription last month and want my money back immediately.",
    "The app keeps crashing every time I try to export a PDF report on Windows 11."
]

for ticket in tickets:
    try:
        result = app.invoke({
            "ticket_text": ticket,
            "intent": "",
            "response": "",
            "actions": [],
            "log": []
        })
        render_support_chat(result["log"], ticket)
    except Exception as e:
        print(f"Error: {e}")
